# Exploratory data analysis (EDA) of the MRI data

Makes the figures and numbers of the EDA report (`docs/EDA_REPORT.md`): how many patients and scans each cohort has,
what the four MRI types look like, tumor sizes and composition, class imbalance, where tumors are in the brain,
intensities before and after cleaning, train vs validation, and difficult cases.

Only **training and validation** scans are opened. Locked test patients and the external cohorts are counted from the
split file but never read.

**How to run:** a CPU runtime is enough (no GPU needed). **Runtime → Run all.** About 1–1.5 hours with MU and RHUH,
about 30 minutes with BraTS 2021 only. Results go to `BrainMRI_Data/eda/` on Drive (figures in `eda/figures/`).

In [ ]:
# 1. Settings and Drive
BRANCH = "main"
CLEAN_MU_RHUH = True       # also clean the MU-Glioma-Post and RHUH-GBM train/val scans (adds ~40-60 min); False = BraTS 2021 only
INTENSITY_SAMPLE = 150     # scans per cohort used for the intensity statistics

import sys, time, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
CLEANED_TAR = BASE / "processed/brats2021_trainval_v1.tar"
OUT = BASE / "eda"
!df -h /content | tail -1

In [ ]:
# 2. Code from the team repo, and the packages
if Path("/content/repo/.git").exists():
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo log --oneline -1
!pip -q install "monai>=1.4" nibabel

In [ ]:
# 3. Cleaned BraTS 2021 training + validation scans in /content/processed/brats2021 (restored from the Drive archive)
processed = Path("/content/processed/brats2021")
start = time.time()
if not processed.exists():
    !mkdir -p /content/processed && tar -xf "{CLEANED_TAR}" -C /content/processed
print(f"{len(list(processed.glob('BraTS2021_*')))} BraTS 2021 patients ready ({(time.time() - start) / 60:.0f} min)")

In [ ]:
# 4. Optional: clean the MU and RHUH training + validation scans with Yasmin's pipeline (test patients stay locked)
DATASETS = ["brats2021"]
if CLEAN_MU_RHUH:
    try:
        sys.path.insert(0, str(BASE / "pipeline"))
        import standardize as S
        start = time.time()
        jobs = S.regenerate(splits=("train", "val"), datasets=("mu", "rhuh"))
        print(jobs.status.value_counts().to_dict(), f"({(time.time() - start) / 60:.0f} min)")
        DATASETS += ["mu", "rhuh"]
    except Exception as e:  # the report still works with BraTS 2021 only
        print("MU/RHUH cleaning failed, continuing with BraTS 2021 only:", repr(e))
DATASET_ARGS = " ".join(DATASETS)
print("cohorts in the EDA:", DATASET_ARGS)

In [ ]:
# 5. Run the EDA: tables, summary.json and figures in BrainMRI_Data/eda
%cd /content/repo
!python scripts/eda.py --out "{OUT}" --datasets {DATASET_ARGS} --intensity-sample {INTENSITY_SAMPLE}

In [ ]:
# 6. Look at the results
from IPython.display import Image, display
summary = json.loads((OUT / "summary.json").read_text())
print(json.dumps({k: summary[k] for k in ("patients", "train_vs_val_ks_p", "spearman_ed_vs_tc_brats2021") if k in summary}, indent=1))
for png in sorted((OUT / "figures").glob("*.png")):
    print(png.name)
    display(Image(str(png), width=1000))

In [ ]:
# 7. Finish writing to Drive
drive.flush_and_unmount()